# 🪶 Automated Feather Measurement and Analysis Pipeline
This notebook runs a **minimal slice** of the distributed pipeline from a Jupyter session while keeping dataset storage and heavy inference on the Mac mini cluster. The notebook acts as the control plane: select one source image, dispatch processing, and review returned feather crops.


## Step 1: Laboratory Equipment Setup (Environment & Cluster Configuration)
This step resolves the project root, configures cluster endpoints, sets Celery broker/backend variables, and prepares a local preview directory. Use environment variables to override hostnames, SSH keys, and remote data paths without editing notebook code.


In [ ]:
import os
import shlex
import subprocess
import sys
import time
from pathlib import Path

from IPython.display import display
from PIL import Image

def detect_repo_root() -> Path:
    env_root = os.getenv('FEATHER_REPO_ROOT', '').strip()
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / 'src' / 'celery_tasks.py').exists():
            return p

    candidates = [Path.cwd(), *Path.cwd().parents]
    for c in candidates:
        if (c / 'src' / 'celery_tasks.py').exists():
            return c
    raise RuntimeError('Could not find repo root containing src/celery_tasks.py. Set FEATHER_REPO_ROOT.')

REPO_ROOT = detect_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Cluster + path config (override with env vars as needed).
HEAD_HOST = os.getenv('FEATHER_HEAD_HOST', '10.0.0.148')
SSH_USER = os.getenv('FEATHER_SSH_USER', 'openteams')
KEY_PATH = os.path.expanduser(os.getenv('FEATHER_SSH_KEY', '~/.ssh/ubuntu-mac-openteams-admin'))
REMOTE_REPO_DIR = os.getenv('FEATHER_REMOTE_REPO_DIR', '/Users/openteams/Feather_Molt_Project')
REMOTE_INPUT_DIR = os.getenv('FEATHER_REMOTE_INPUT_DIR', f'{REMOTE_REPO_DIR}/data/raw')
REMOTE_OUTPUT_DIR = os.getenv('FEATHER_REMOTE_OUTPUT_DIR', f'{REMOTE_REPO_DIR}/data/processed')
CLUSTER_HOSTS = [h.strip() for h in os.getenv('FEATHER_CLUSTER_HOSTS', '10.0.0.148,10.0.0.63,10.0.0.19,10.0.0.118').split(',') if h.strip()]

# Celery broker/backend reachable from this notebook host.
os.environ['BROKER_URL'] = os.getenv('BROKER_URL', f'redis://{HEAD_HOST}:6379/0')
os.environ['RESULT_BACKEND'] = os.getenv('RESULT_BACKEND', f'redis://{HEAD_HOST}:6379/1')

LOCAL_PREVIEW_DIR = Path('./.minimal_slice_preview').resolve()
LOCAL_PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT =', REPO_ROOT)
print('HEAD_HOST =', HEAD_HOST)
print('CLUSTER_HOSTS =', CLUSTER_HOSTS)
print('BROKER_URL =', os.environ['BROKER_URL'])
print('REMOTE_INPUT_DIR =', REMOTE_INPUT_DIR)
print('REMOTE_OUTPUT_DIR =', REMOTE_OUTPUT_DIR)
print('LOCAL_PREVIEW_DIR =', LOCAL_PREVIEW_DIR)


## Step 2: Remote Inventory and Transport Helpers
Helper functions provide cluster-safe file discovery and transfer operations over SSH/SCP. This enables image discovery and result collection directly from cluster nodes rather than mirroring the full dataset locally.


In [ ]:
def _ssh_base_cmd(host: str):
    cmd = ['ssh']
    if KEY_PATH:
        cmd += ['-i', KEY_PATH]
    cmd += ['-o', 'StrictHostKeyChecking=no', f'{SSH_USER}@{host}']
    return cmd

def ssh_lines(host: str, remote_cmd: str):
    out = subprocess.check_output([*_ssh_base_cmd(host), remote_cmd], text=True)
    return [line.strip() for line in out.splitlines() if line.strip()]

def list_remote_images(input_host: str = HEAD_HOST):
    d = shlex.quote(REMOTE_INPUT_DIR)
    cmd = f"find {d} -maxdepth 1 -type f \\( -iname '*.jpg' -o -iname '*.jpeg' \\) | sort"
    return ssh_lines(input_host, cmd)

def list_remote_outputs(host: str):
    d = shlex.quote(REMOTE_OUTPUT_DIR)
    cmd = f"find {d} -maxdepth 1 -type f -iname '*.jpg' | sort"
    return ssh_lines(host, cmd)

def fetch_remote_file(host: str, remote_path: str, local_dir: Path):
    local_dir.mkdir(parents=True, exist_ok=True)
    remote_target = f"{SSH_USER}@{host}:{shlex.quote(remote_path)}"
    cmd = ['scp']
    if KEY_PATH:
        cmd += ['-i', KEY_PATH]
    cmd += ['-o', 'StrictHostKeyChecking=no', remote_target, str(local_dir)]
    subprocess.check_call(cmd)


## Step 3: Minimal Slice Dispatch (Single-Image Distributed Processing)
A single source image is selected from the remote input directory and submitted as a Celery task. Workers perform Grounding DINO + SAM extraction and metadata handling on the cluster, then return task status to this notebook.

**Integration Note:** this is the insertion point for broader batch runs or alternate orchestration, while preserving the same remote execution model.


In [ ]:
from src.celery_tasks import process_image

remote_images = list_remote_images(HEAD_HOST)
print(f'Found {len(remote_images)} remote images on {HEAD_HOST}.')
if not remote_images:
    raise RuntimeError('No remote images found. Check FEATHER_REMOTE_INPUT_DIR.')

SAMPLE_INDEX = 0  # Change this if you want a different source image.
image_path = remote_images[SAMPLE_INDEX]
source_stem = Path(image_path).stem
print('Selected remote source:', image_path)

async_result = process_image.delay(image_path, REMOTE_OUTPUT_DIR)
print('Task ID:', async_result.id)

while not async_result.ready():
    print('Waiting for worker completion...')
    time.sleep(3)

result = async_result.get(timeout=10)
print('Task result:', result)


## Step 4: Retrieval and Visual Review
After task completion, this step searches cluster nodes for generated feather crops, pulls matching outputs to a local preview directory, and renders them inline for QA.


In [ ]:
prefix = f"{source_stem}_"
matching = []

for host in CLUSTER_HOSTS:
    try:
        out_files = list_remote_outputs(host)
    except Exception as exc:
        print(f'Skipping host {host}: {exc}')
        continue

    host_matches = []
    for p in out_files:
        name = Path(p).name
        if name.startswith(prefix) and '_Feather_' in name:
            host_matches.append(p)

    if host_matches:
        print(f'Found {len(host_matches)} segment(s) on {host}.')
        matching = [(host, p) for p in host_matches]
        break

if not matching:
    raise RuntimeError('No matching segment outputs found across cluster hosts.')

preview_dir = LOCAL_PREVIEW_DIR / source_stem
preview_dir.mkdir(parents=True, exist_ok=True)

for host, remote_file in matching:
    fetch_remote_file(host, remote_file, preview_dir)

local_files = sorted(preview_dir.glob('*.jpg'))
print(f'Fetched {len(local_files)} files into {preview_dir}')

for f in local_files:
    print(f.name)
    display(Image.open(f))
